# OLMo 3 7B Think Budget Sweep

Purpose: measure forced-close rate, accuracy, output length, and runtime for `allenai/Olmo-3-7B-Think` under fixed thinking budgets on a small standard-format Goal 1 dataset. This notebook is designed for a Colab A100 runtime.

The dataset and experiment configs are defined inside the notebook. The repo is cloned or pulled only for package code.

## 1. Clone Or Update Repo

Edit `REPO_URL` if the GitHub remote is different or private. If the repo is private, authenticate with Colab/GitHub before running this cell.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/ndalton12/carry-trace.git"  # Change if needed.
BRANCH = "main"
REPO_DIR = Path("/content/carry-trace")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print(f"Repo: {REPO_DIR}")
print(f"Commit: {commit}")
print(f"Python: {sys.version}")

## 2. Install Dependencies

Colab usually already has CUDA PyTorch. This installs the repo package without reinstalling PyTorch, then installs the non-Torch dependencies plus bitsandbytes for optional 8-bit benchmarking.

In [ ]:
!python -m pip install -U pip
!python -m pip install -e . --no-deps --ignore-requires-python
!python -m pip install "accelerate>=1.0.0" "bitsandbytes>=0.45.0" "datasets>=3.0.0" "matplotlib>=3.9.0" "numpy>=2.0.0" "pandas>=2.2.0" "pyarrow>=17.0.0" "pydantic>=2.9.0" "pyyaml>=6.0.2" "rich>=13.8.0" "seaborn>=0.13.2" "transformers>=4.45.0" "typer>=0.12.5" "huggingface_hub[hf_transfer]>=0.23.0"

In [ ]:
import os
import sys

os.chdir(REPO_DIR)
src_path = str(REPO_DIR / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import carry_trace

print("carry_trace imported from", carry_trace.__file__)

## 3. GPU And Hugging Face Setup

Set `HF_TOKEN` in Colab secrets or `os.environ` if Hugging Face auth is required in your environment.

In [ ]:
import os
import torch

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))
    print("bf16 supported", torch.cuda.is_bf16_supported())

!nvidia-smi

In [ ]:
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged into Hugging Face using HF_TOKEN.")
else:
    print("No HF_TOKEN set. Continuing unauthenticated.")

## 4. Define Sweep Configs

This uses standard input and standard output only. The goal is to isolate the effect of thinking budget, not format ablations. The dataset has 48 examples by default: 4 digit lengths x 6 slices x 2 prompt modes x 1 replicate.

In [ ]:
from pathlib import Path
import yaml

SWEEP_NAME = "colab_olmo3_think_budget_sweep"
SEED = 20260610
EXAMPLES_PER_SLICE_PER_LENGTH = 1
BUDGETS = [512, 1024, 2048]
THINKING_FINAL_ANSWER_TOKENS = 100
BATCH_SIZE = 4
QUANTIZATION = "none"  # Set to "bitsandbytes_8bit" for a memory/throughput comparison.

DIGIT_LENGTHS = [2, 4, 8, 12]
SLICES = [
    "no_carry",
    "isolated_carry",
    "long_carry_chain",
    "internal_carry_chain",
    "carry_distractor",
    "many_9s_no_carry",
]
PROMPT_MODES = ["answer_only", "free_cot"]
DIGIT_FORMATS = ["standard"]
ANSWER_FORMATS = ["standard"]

config_dir = Path("configs/colab")
config_dir.mkdir(parents=True, exist_ok=True)

dataset_config = {
    "name": SWEEP_NAME,
    "seed": SEED,
    "base": 10,
    "output_dir": "data/generated",
    "write_parquet": False,
    "schema_version": "goal1.v1",
    "splits": {
        "budget_sweep": {
            "examples_per_slice_per_length": EXAMPLES_PER_SLICE_PER_LENGTH,
        },
    },
    "digit_lengths": DIGIT_LENGTHS,
    "slices": SLICES,
    "prompt_modes": PROMPT_MODES,
    "digit_formats": DIGIT_FORMATS,
    "answer_formats": ANSWER_FORMATS,
    "digit_delimiter": "|",
}

dataset_config_path = config_dir / "dataset_budget_sweep.yaml"
dataset_config_path.write_text(yaml.safe_dump(dataset_config, sort_keys=False), encoding="utf-8")

experiment_config_paths = []
for budget in BUDGETS:
    experiment_config = {
        "name": f"{SWEEP_NAME}_{budget}",
        "seed": SEED,
        "dataset_path": f"data/generated/{SWEEP_NAME}/examples.jsonl",
        "output_dir": "runs",
        "splits": ["budget_sweep"],
        "prompt_modes": PROMPT_MODES,
        "digit_formats": DIGIT_FORMATS,
        "answer_formats": ANSWER_FORMATS,
        "runner": {
            "kind": "hf",
            "device": "auto",
            "dtype": "bfloat16",
            "batch_size": BATCH_SIZE,
            "trust_remote_code": False,
            "quantization": QUANTIZATION,
        },
        "models": [
            {
                "name": "olmo3-think",
                "model_id": "allenai/Olmo-3-7B-Think",
            },
        ],
        "generation": {
            "max_new_tokens": budget,
            "temperature": 0.0,
            "top_p": 1.0,
            "do_sample": False,
            "thinking_final_answer_tokens": THINKING_FINAL_ANSWER_TOKENS,
            "force_close_thinking": True,
        },
    }
    path = config_dir / f"experiment_budget_{budget}.yaml"
    path.write_text(yaml.safe_dump(experiment_config, sort_keys=False), encoding="utf-8")
    experiment_config_paths.append(path)

expected_rows = (
    len(DIGIT_LENGTHS)
    * len(SLICES)
    * EXAMPLES_PER_SLICE_PER_LENGTH
    * len(PROMPT_MODES)
)
print(f"Expected examples per budget: {expected_rows}")
print("\nDataset config:\n", dataset_config_path.read_text())
for path in experiment_config_paths:
    print("\nExperiment config:", path)
    print(path.read_text())

## 5. Generate Dataset

In [ ]:
!carry-trace dataset generate --config configs/colab/dataset_budget_sweep.yaml

In [ ]:
from collections import Counter
from pathlib import Path
from carry_trace.io import read_jsonl

examples = read_jsonl(Path(f"data/generated/{SWEEP_NAME}/examples.jsonl"))
print("rows", len(examples))
print("n_digits", Counter(row["n_digits"] for row in examples))
print("slice_name", Counter(row["slice_name"] for row in examples))
print("prompt_mode", Counter(row["prompt_mode"] for row in examples))
print("digit_format", Counter(row["digit_format"] for row in examples))
print("answer_format", Counter(row["answer_format"] for row in examples))
print("\nFirst prompt:\n", examples[0]["prompt"])
print("\nFirst expected_output:", examples[0]["expected_output"])

## 6. Run Budget Sweep

This runs the same dataset once per budget. The printed wall time includes model loading for each budget because the CLI starts a fresh process. Per-example `latency_seconds` in `calls.jsonl` is measured after the model is loaded.

In [ ]:
import subprocess
import time
from pathlib import Path

run_dirs_by_budget = {}
for budget, config_path in zip(BUDGETS, experiment_config_paths, strict=True):
    print("=" * 100)
    print("Running budget", budget)
    before = set(Path("runs").glob(f"{SWEEP_NAME}_{budget}-*"))
    started = time.perf_counter()
    process = subprocess.Popen(
        ["carry-trace", "run", "goal1", "--config", str(config_path)],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    wall_seconds = time.perf_counter() - started
    if return_code != 0:
        raise RuntimeError(f"carry-trace run failed for budget {budget} with exit code {return_code}")
    after = set(Path("runs").glob(f"{SWEEP_NAME}_{budget}-*"))
    new_dirs = sorted(after - before, key=lambda path: path.stat().st_mtime)
    if not new_dirs:
        new_dirs = sorted(Path("runs").glob(f"{SWEEP_NAME}_{budget}-*"), key=lambda path: path.stat().st_mtime)
    run_dirs_by_budget[budget] = new_dirs[-1]
    print("RUN_DIR", run_dirs_by_budget[budget])
    print("wall_seconds", round(wall_seconds, 2))

## 7. Summarize Forced-Close Rate, Accuracy, And Runtime

In [ ]:
import pandas as pd
from carry_trace.io import read_jsonl

summary_rows = []
for budget, run_dir in run_dirs_by_budget.items():
    calls = read_jsonl(run_dir / "calls.jsonl")
    scored = read_jsonl(run_dir / "scored_calls.jsonl")
    forced = [bool(row.get("metadata", {}).get("thinking_force_closed")) for row in calls]
    total_output_tokens = sum(row["token_count_output"] for row in calls)
    sum_generation_latency = sum(row["latency_seconds"] for row in calls)
    summary_rows.append(
        {
            "budget": budget,
            "examples": len(calls),
            "forced_close_rate": sum(forced) / len(forced),
            "forced_close_count": sum(forced),
            "parsed_accuracy": sum(row["parsed_answer_correct"] for row in scored) / len(scored),
            "output_format_accuracy": sum(row["parsed_output_format_correct"] for row in scored) / len(scored),
            "avg_output_tokens": total_output_tokens / len(calls),
            "sum_generation_latency_seconds": sum_generation_latency,
            "examples_per_second_after_load": len(calls) / sum_generation_latency,
            "tokens_per_second_after_load": total_output_tokens / sum_generation_latency,
            "run_dir": str(run_dir),
        }
    )

summary = pd.DataFrame(summary_rows).sort_values("budget")
display(summary)
summary.to_csv("runs/olmo3_think_budget_sweep_summary.csv", index=False)
print("Wrote runs/olmo3_think_budget_sweep_summary.csv")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(summary["budget"], summary["forced_close_rate"], marker="o")
axes[0].set_title("Forced-Close Rate")
axes[0].set_xlabel("max_new_tokens")
axes[0].set_ylim(0, 1)

axes[1].plot(summary["budget"], summary["parsed_accuracy"], marker="o", label="parsed")
axes[1].plot(summary["budget"], summary["output_format_accuracy"], marker="o", label="format")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("max_new_tokens")
axes[1].set_ylim(0, 1)
axes[1].legend()

axes[2].plot(summary["budget"], summary["examples_per_second_after_load"], marker="o")
axes[2].set_title("Throughput")
axes[2].set_xlabel("max_new_tokens")
axes[2].set_ylabel("examples/sec")

plt.tight_layout()
plt.show()

## 8. Inspect Forced-Close Examples

This writes full decoded outputs for each budget so Colab output truncation does not hide tails.

In [ ]:
for budget, run_dir in run_dirs_by_budget.items():
    calls = read_jsonl(run_dir / "calls.jsonl")
    scored = read_jsonl(run_dir / "scored_calls.jsonl")
    examples_by_id = {row["id"]: row for row in read_jsonl(run_dir / "dataset.jsonl")}
    sections = []
    for idx, row in enumerate(scored, start=1):
        call = calls[idx - 1]
        example = examples_by_id[row["example_id"]]
        call_metadata = row.get("metadata", {})
        section = "
".join(
            [
                "=" * 100,
                f"Budget {budget} example {idx}/{len(scored)} example_id={row['example_id']}",
                f"example_metadata: {{'n_digits': {example['n_digits']}, 'slice_name': {example['slice_name']!r}, 'prompt_mode': {example['prompt_mode']!r}}}",
                f"call_metadata: {call_metadata}",
                f"token_count_output: {row.get('token_count_output')}",
                f"parsed_answer_correct: {row.get('parsed_answer_correct')}",
                f"decoded_output:
{row['decoded_output']}",
            ]
        )
        sections.append(section)
    output_path = run_dir / "full_decoded_outputs.txt"
    output_path.write_text("

".join(sections), encoding="utf-8")
    forced_count = sum(bool(row.get("metadata", {}).get("thinking_force_closed")) for row in scored)
    print(f"budget={budget} forced_close_count={forced_count} wrote {output_path}")

# Print just the first forced-close example per budget, if any.
for budget, run_dir in run_dirs_by_budget.items():
    scored = read_jsonl(run_dir / "scored_calls.jsonl")
    forced_rows = [row for row in scored if row.get("metadata", {}).get("thinking_force_closed")]
    if forced_rows:
        print("=" * 100)
        print("First forced-close example for budget", budget)
        print(forced_rows[0]["decoded_output"])